# Inspect aligned training samples

Shows what the detector is trained on: real / full re-decode / local paste, with the soft target map.
Run from the repository root.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '..') if 'trace_detector' not in sys.modules else None
from trace_detector.train_detector import load_config
from trace_detector.data import AlignedPairsDataset

cfg = load_config('../trace_detector/configs/train_config_detector.yaml')
cfg['data']['pairs_csv'] = '../' + cfg['data']['pairs_csv']
dataset = AlignedPairsDataset(cfg['data']['pairs_csv'], cfg['data'], split='train')
len(dataset)

In [ ]:
def show(n=6, seed=0):
    dataset.seed = seed
    fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
    for i in range(n):
        item = dataset[np.random.default_rng(seed + i).integers(len(dataset))]
        axes[i, 0].imshow(item['image'].permute(1, 2, 0).numpy()); axes[i, 0].set_title(f"{item['mode']}  label={item['label']:.2f}")
        axes[i, 1].imshow(item['target'][0].numpy(), vmin=0, vmax=1, cmap='magma'); axes[i, 1].set_title('target map')
        for ax in axes[i]: ax.axis('off')
    plt.tight_layout()

show()

## Heat maps of a trained model

Point `weights` at `best.pt` and `paths` at a few images.

In [ ]:
from PIL import Image
from trace_detector.model_detector_torch import TraceDetectorInference

weights = '../checkpoints/trace_dinov3_b16/best.pt'
paths = []  # e.g. ['../data/raw/dailybench/.../x.png']
if paths:
    detector = TraceDetectorInference(weights, device='cuda' if torch.cuda.is_available() else 'cpu')
    fig, axes = plt.subplots(len(paths), 2, figsize=(8, 4 * len(paths)), squeeze=False)
    for ax_row, p in zip(axes, paths):
        r = detector.predict(Image.open(p))
        ax_row[0].imshow(Image.open(p).convert('RGB')); ax_row[0].set_title(f"score {r['score']:.3f}")
        ax_row[1].imshow(r['map'], vmin=0, vmax=1, cmap='magma'); ax_row[1].set_title('generated-region map')
        for ax in ax_row: ax.axis('off')
    plt.tight_layout()